# Notebook 13: Model Evaluation and Selection
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Understand why a held-out test set is essential
- Apply k-fold and stratified cross-validation correctly
- Choose the right evaluation metric for the task
- Read and interpret confusion matrices
- Use ROC-AUC and Precision-Recall curves
- Tune hyperparameters without data leakage

## 1. The Train / Validation / Test Split

| Split | Purpose |
|-------|---------|
| **Train** | Fit model parameters |
| **Validation** | Select hyperparameters / model type |
| **Test** | Unbiased final performance estimate |

**Golden rule**: the test set must never influence any modelling decision. Once you peek at it, the estimate is biased.

### Cross-Validation
With small datasets, a single val split is too noisy. **k-fold CV** gives k estimates:
1. Split data into k equal folds
2. For fold i: train on the other k−1 folds, validate on fold i
3. Average the k scores

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    GridSearchCV, learning_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    classification_report
)
import warnings; warnings.filterwarnings('ignore')

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f'Train: {X_tr.shape}, Test: {X_te.shape}')

In [ ]:
pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X_tr, y_tr, cv=skf, scoring='accuracy')
print('5-fold CV scores:', scores.round(4))
print(f'Mean: {scores.mean():.4f} ± {scores.std():.4f}')

## 2. Classification Metrics

Accuracy is not always the right metric.

**Imbalanced example**: 99% of transactions are non-fraudulent. A classifier that always says "not fraud" gets 99% accuracy but is useless.

| Metric | Formula | Use when |
|--------|---------|----------|
| **Accuracy** | (TP+TN)/(P+N) | Balanced classes |
| **Precision** | TP/(TP+FP) | FP cost is high (spam filter) |
| **Recall** | TP/(TP+FN) | FN cost is high (cancer screening) |
| **F1** | 2·P·R/(P+R) | Need balance of P and R |
| **ROC-AUC** | Area under ROC curve | Ranking quality across thresholds |

In [ ]:
pipe.fit(X_tr, y_tr)
y_pred = pipe.predict(X_te)
y_prob = pipe.predict_proba(X_te)[:, 1]

print(f'Accuracy : {accuracy_score(y_te, y_pred):.4f}')
print(f'Precision: {precision_score(y_te, y_pred):.4f}')
print(f'Recall   : {recall_score(y_te, y_pred):.4f}')
print(f'F1       : {f1_score(y_te, y_pred):.4f}')
print()
print(classification_report(y_te, y_pred, target_names=cancer.target_names))

In [ ]:
cm = confusion_matrix(y_te, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_xticklabels(cancer.target_names)
axes[0].set_yticks([0,1]); axes[0].set_yticklabels(cancer.target_names)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        lbl = ['TN','FP','FN','TP'][i*2+j]
        axes[0].text(j, i, f'{lbl}\n{cm[i,j]}', ha='center', va='center',
                     color='white' if cm[i,j]>cm.max()/2 else 'black', fontsize=11)

fpr, tpr, _ = roc_curve(y_te, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, linewidth=2, label=f'Logistic Reg (AUC={roc_auc:.3f})')
axes[1].plot([0,1],[0,1], 'k--', alpha=0.4)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve'); axes[1].legend()
plt.tight_layout(); plt.show()

## 3. Precision-Recall Curve

For highly imbalanced classes, ROC can be misleadingly optimistic. The **Precision-Recall curve** gives a clearer picture.

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_te, y_prob)
avg_prec = average_precision_score(y_te, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(recalls, precisions, linewidth=2)
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title(f'Precision-Recall (AP={avg_prec:.3f})')

axes[1].plot(thresholds, precisions[:-1], label='Precision')
axes[1].plot(thresholds, recalls[:-1],    label='Recall')
axes[1].axvline(0.5, color='red', linestyle='--', label='threshold=0.5')
axes[1].set_xlabel('Threshold'); axes[1].set_title('P & R vs Threshold')
axes[1].legend()
plt.tight_layout(); plt.show()

## 4. Learning Curves — Diagnosing Bias and Variance

- **High bias (underfitting)**: both train and val error are high
- **High variance (overfitting)**: train error is low, large gap to val error
- **Good fit**: both converge to low error with enough data

In [ ]:
def plot_learning_curve(model, X, y, title, ax):
    sizes, tr_scores, val_scores = learning_curve(
        model, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy', n_jobs=-1
    )
    ax.plot(sizes, tr_scores.mean(1),  marker='o', label='Train')
    ax.plot(sizes, val_scores.mean(1), marker='s', label='Validation')
    ax.fill_between(sizes, tr_scores.mean(1)-tr_scores.std(1),
                           tr_scores.mean(1)+tr_scores.std(1), alpha=0.15)
    ax.fill_between(sizes, val_scores.mean(1)-val_scores.std(1),
                           val_scores.mean(1)+val_scores.std(1), alpha=0.15)
    ax.set_xlabel('Training samples'); ax.set_ylabel('Accuracy')
    ax.set_title(title); ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_learning_curve(
    Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=1000))]),
    X, y, 'Logistic Regression', axes[0])
plot_learning_curve(
    DecisionTreeClassifier(max_depth=None),
    X, y, 'Unconstrained Decision Tree', axes[1])
plt.tight_layout(); plt.show()

## 5. Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'clf__C'       : [0.001, 0.01, 0.1, 1, 10],
    'clf__penalty' : ['l1', 'l2'],
}
pipe_lr = Pipeline([('sc', StandardScaler()),
                     ('clf', LogisticRegression(max_iter=1000, solver='liblinear'))])
grid = GridSearchCV(pipe_lr, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_tr, y_tr)

print(f'Best params: {grid.best_params_}')
print(f'Best CV F1 : {grid.best_score_:.4f}')
print(f'Test F1    : {f1_score(y_te, grid.best_estimator_.predict(X_te)):.4f}')

## Exercises

1. Create a synthetic imbalanced dataset (95% class 0, 5% class 1). Compare Accuracy, F1, and ROC-AUC. Which is most informative?
2. Use `RandomizedSearchCV` instead of `GridSearchCV` on a Random Forest. Is it faster? Does it find a comparable result?
3. Plot ROC curves for four classifiers on the same axes. Which model dominates?
4. Implement **nested cross-validation** (outer loop for performance estimation, inner loop for tuning). When is this necessary?

## Reflection Questions
- You tune hyperparameters on the validation fold and then report test performance. Is this estimate still unbiased? Why?
- What does ROC-AUC = 0.5 mean? What about 1.0?
- Precision and Recall trade off against each other. In a spam filter, which matters more? In a cancer screening test?

---
**Next →** `14_clustering.ipynb`